# Kaggle / Colab runner

Clones the repo and runs the project scripts on a free GPU.

**Choose the phase** with `PHASE` in the first code cell: `3` = baseline with epoch-budget search, `4` = the baseline with 3 seeds plus the skin-tone analysis, `5` = the four image-processing settings (3 seeds each) and their comparison, `6` = the final baseline model and its Grad-CAM heatmaps.

**Kaggle settings (right-hand panel):** Accelerator → *GPU T4 x2* or *GPU P100*; Internet → *On*
(needs a phone-verified account).

**Run it with *Save Version → Save & Run All (Commit)*,** not interactively. That runs in the background (up to 12 h, no idle timeout) and keeps everything in `/kaggle/working` as the version's output.

The repo and data live in a temporary folder (`/kaggle/temp`), so the ~3 GB of images are not saved as notebook output.
Each finished budget trial and CV fold is saved to `/kaggle/working/baseline_folds` (Phase 3) `/kaggle/working/seed_folds` (Phase 4) or `/kaggle/working/phase5_folds/<setting>` (Phase 5) straight away, so a crash or timeout keeps the finished folds. To resume, add the previous version's output as an input and set `RESUME_FROM` below; the training script skips the folds already saved.

Results are zipped into `/kaggle/working/phase<PHASE>_outputs.zip` at the end: download it and put it in the repo root.

**Get results by downloading, not by syncing.** Download the zip from the version's *Output* tab. Keep Kaggle's "Save to GitHub" / GitHub sync **off** for this notebook: it commits the executed copy (with all outputs) straight to `main`, which then conflicts with local commits.

In [1]:
import os, subprocess
REPO_URL = "https://github.com/sehersharik/skin-tone-triage.git"
WORK = "/kaggle/temp" if os.path.isdir("/kaggle") else "/content"   # Kaggle or Colab
os.makedirs(WORK, exist_ok=True)
OUTPUT = "/kaggle/working" if os.path.isdir("/kaggle") else "/content"  # survives a timeout on Kaggle
PHASE = 6   # 3 = baseline + epoch budget, 4 = 3 seeds + skin-tone analysis, 5 = image-processing settings,
            # 6 = final model + Grad-CAM
FOLD_DIR = f"{OUTPUT}/" + {3: "baseline_folds", 4: "seed_folds", 5: "phase5_folds", 6: "final_folds"}[PHASE]
RESUME_FROM = None   # e.g. "/kaggle/input/<previous-version-output>/seed_folds"
if RESUME_FROM and not os.path.isdir(FOLD_DIR):
    !cp -r {RESUME_FROM} {FOLD_DIR}
%cd {WORK}
!rm -rf skin-tone-triage
if subprocess.run(["git", "clone", "-q", REPO_URL]).returncode != 0:
    raise RuntimeError("Could not clone the repo. On Kaggle: Settings -> Internet -> On "
                       "(needs a phone-verified account), then run again.")
%cd skin-tone-triage
!git log --oneline -1
!pip install -q -r requirements.txt

/kaggle/temp


/kaggle/temp/skin-tone-triage
cf6a315 (HEAD -> main, origin/main, origin/HEAD) Phase 6 setup: final baseline model and Grad-CAM


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires ipykernel==6.17.1, but you have ipykernel 7.4.0 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.20.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 2.3.3 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
notebook 6.5.7 requires jupyter-client<8,>=5.3.4, but you have jupyter-client 8.10.0 which is incompatible.
jupyter-kernel-gateway 2.5.2 requires jupyter_client<8.0,>=5.2.0, but you have jupyter-client 8.10.0 which is incompatible.


In [2]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB
Tesla T4, 15360 MiB


## Data: metadata, splits, and only the images we use

`build_metadata` stops if the splits differ from the committed `reports/split_fingerprint.txt`.

In [3]:
!python -m src.download_scin --config configs/data.yaml --images none
!python -m src.build_metadata --config configs/labels.yaml
!python -m src.download_scin --config configs/data.yaml --images used
!python -m src.prepare_images --config configs/baseline.yaml

dataset/scin_cases.csv: downloaded


dataset/scin_labels.csv: downloaded


dataset/scin_app_questions.csv: downloaded


dataset/scin_label_questions.csv: downloaded


images: 0it [00:00, ?it/s]
{'image_mode': 'none', 'images_referenced_in_cases_csv': 10380, 'images_requested': 0, 'downloaded': 0, 'skipped_existing': 0} failed=0
Run log: runs/20261103_083344_download_scin


/usr/local/lib/python3.13/dist-packages/sklearn/model_selection/_split.py:1036: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


... (681 outputs left out)

resize:  99%|██████████████████████████████▊| 2514/2534 [01:05<00:00, 34.99it/s]

resize: 100%|██████████████████████████████▊| 2522/2534 [01:05<00:00, 42.58it/s]

resize: 100%|██████████████████████████████▉| 2528/2534 [01:05<00:00, 39.68it/s]

resize: 100%|███████████████████████████████| 2534/2534 [01:05<00:00, 38.74it/s]
{'resized': 2534}


## Phase 3: baseline 5-fold CV (test set untouched) and evaluation

In [4]:
if PHASE == 3:
    !python -m src.train --config configs/baseline.yaml --fold-outputs-dir {FOLD_DIR}
    !python -m src.evaluate --config configs/baseline.yaml

## Phase 4: 3 seeds on the same folds, then the skin-tone analysis (test set untouched)

About 4 min of training per seed on a T4. `image_factors` measures brightness, contrast, sharpness and resolution of every CV photo; `tone_analysis` writes `tone_analysis_cv` to `reports/metrics.json` plus the `phase4_*` figures.

In [5]:
if PHASE == 4:
    !python -m src.train --config configs/seeds.yaml --fold-outputs-dir {FOLD_DIR}
    !python -m src.image_factors --config configs/seeds.yaml
    !python -m src.tone_analysis --config configs/seeds.yaml

## Phase 5: image-processing settings, 3 seeds x 5 folds each (test set untouched)

For each setting the epoch budget is re-picked on fold 1 (seed 42), then 3 seeds x 5 folds are trained. Settings run one after another; every finished budget trial and fold is saved, so a rerun with `RESUME_FROM` continues where it stopped. `compare_settings` writes `phase5_cv` to `reports/metrics.json` and the `phase5_*` figures.

In [6]:
if PHASE == 5:
    for setting in ["none", "standard_aug", "shades_of_gray", "clahe"]:
        !python -m src.train --config configs/phase5/{setting}.yaml --fold-outputs-dir {FOLD_DIR}/{setting}
    !python -m src.compare_settings --config configs/phase5/compare.yaml

## Phase 6: final baseline model and Grad-CAM (test set untouched)

About 15 min on a T4. (1) Seed-42 out-of-fold predictions on the CV pool, which decide which cases are "correct" or "incorrect". (2) One model trained on all 928 CV cases (5 epochs, seed 42), saved to `models/` and included in the zip. (3) Grad-CAM from that model on every CV image. The grids in `reports/gradcam/` show SCIN photos: they are gitignored and must not be committed.

In [7]:
if PHASE == 6:
    !python -m src.train --config configs/final.yaml --fold-outputs-dir {FOLD_DIR}
    !python -m src.train_final --config configs/final.yaml
    !python -m src.gradcam --config configs/final.yaml

Device: cuda  cases: 928  images: 2054  classes: ['dermatitis', 'folliculitis', 'insect_bite', 'psoriasis', 'tinea', 'urticaria']
CV fold 0: 5 epochs


Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


  0%|                                               | 0.00/20.5M [00:00<?, ?B/s]

100%|███████████████████████████████████████| 20.5M/20.5M [00:00<00:00, 188MB/s]


    epoch 1/5  train loss 1.7226


    epoch 2/5  train loss 1.1544


... (627 outputs left out)

Grad-CAM: 100%|████████████████████████████▉| 2047/2054 [01:10<00:00, 29.57it/s]

Grad-CAM: 100%|████████████████████████████▉| 2051/2054 [01:10<00:00, 29.88it/s]

Grad-CAM: 100%|█████████████████████████████| 2054/2054 [01:10<00:00, 29.18it/s]


{
  "border_share_if_uniform": 0.3543,
  "border_share_by_group": {
    "all": {
      "n_cases": 928,
      "mean": 0.1934,
      "ci_low": 0.1855,
      "ci_high": 0.2018
    },
    "FST I-II, correct": {
      "n_cases": 176,
      "mean": 0.1369,
      "ci_low": 0.1241,
      "ci_high": 0.1506
    },
    "FST I-II, incorrect": {
      "n_cases": 226,
      "mean": 0.2329,
      "ci_low": 0.2154,
      "ci_high": 0.2529
    },
    "FST III-IV, correct": {
      "n_cases": 211,
      "mean": 0.1449,
      "ci_low": 0.134,
      "ci_high": 0.1561
    },
    "FST III-IV, incorrect": {
      "n_cases": 214,
      "mean": 0.2523,
      "ci_low": 0.2318,
      "ci_high": 0.272
    },
    "FST V-VI, correct": {
      "n_cases": 35,
      "mean": 0.1425,
      "ci_low": 0.1155,
      "ci_high": 0.1699
    },
    "FST V-VI, incorrect": {
      "n_cases": 26,
      "mean": 0.2505,
      "ci_low": 0.2074,
      "ci_high": 0.296
    }
  },
  "share_of_images_peaking_in_border": 0.1811
}
Wrote g

In [8]:
!pytest -q -p no:warnings
OUT = f"{OUTPUT}/phase{PHASE}_outputs.zip"
EXTRA = "models" if PHASE == 6 else ""   # the final model weights (~16 MB), needed for the app
!zip -qr {OUT} reports docs/figures runs {EXTRA}
import glob, json
for log in sorted(glob.glob({3: "reports/baseline_train_log.json", 4: "reports/seeds/*_train_log.json",
                                5: "reports/phase5/*_train_log.json",
                                6: "reports/final/*_train_log.json"}[PHASE])):
    t = json.load(open(log))["timing"]
    print(log, "| GPU:", t["gpu"], "| seconds per CV fold:", t["seconds_per_cv_fold"])
print("Download:", OUT)

....

...

.

.

..

....

... (5 outputs left out)

....

.....

.                         [100%]
48 passed in 31.14s


reports/final/baseline_seed42_train_log.json | GPU: Tesla T4 | seconds per CV fold: {'0': 46.6, '1': 43.1, '2': 45.1, '3': 45.9, '4': 45.5}
Download: /kaggle/working/phase6_outputs.zip
